# GLM-OCR — LoRA fine-tuning for Tunisian bills of exchange

Same architecture and LoRA configuration as the invoice model, trained to extract seven fields from bills of exchange
(*lettres de change*, كمبيالة): `numero_traite`, `date_echeance`, `montant`, `tireur`, `tire`, `rib`, `domiciliation`.

| Setting | Value |
|---|---|
| LoRA | r = 4, α = 8, dropout 0.2 (2,727,936 trainable parameters) |
| Optimiser | AdamW (fused), FP16, cosine schedule, weight decay 0.05, NEFTune α = 5 |
| Learning rate | 5e-6, lowered to 2e-6 in later sessions |
| Early stopping | patience 20 epochs on evaluation loss |
| Data | 4,258 training / 1,065 test documents |
| Best evaluation loss | 0.0352 |

This notebook is the **last resumed session** (training is spread over several GPU sessions of at most 12 h).
The example values given in the prompt were replaced with fictitious ones; the original prompt used values taken
from a real document.

> **Environment.** Kaggle notebook, NVIDIA Tesla T4 (16 GB), Python 3.12. GLM-OCR requires the development
> version of `transformers` (installed in the first cell). After the first cell, **restart the kernel**.
>
> **Data.** The datasets are private (client documents) and are not distributed. Paths of the form
> `/kaggle/input/datasets/<KAGGLE_USER>/...` refer to private Kaggle datasets and must be adapted.

> **Anonymisation.** Outputs containing client or supplier data and document images were removed.
> Only aggregate metrics are kept.

> **Language.** Comments and messages were translated into English after the run. The prompt is kept in French (with fictitious example values), as used for training.


## 1. Environment setup (restart the kernel afterwards)

In [ ]:
import os, sys, subprocess, shutil, site
from pathlib import Path

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["OMP_NUM_THREADS"]        = "1"

def pip(cmd):
    r = subprocess.run([sys.executable, "-m", "pip"] + cmd,
                       capture_output=True, text=True)
    if r.returncode != 0:
        print("ERR:", r.stderr[-300:])
    return r

pip(["uninstall", "-y", "transformers", "tokenizers",
     "huggingface-hub", "bitsandbytes", "peft", "trl", "accelerate"])

r = pip(["install", "-q", "--no-cache-dir",
         "git+https://github.com/huggingface/transformers.git"])
print(("OK" if r.returncode==0 else "ERR"), "transformers @ git")

for p in ["huggingface-hub==0.28.1", "accelerate==1.3.0",
          "peft==0.13.2", "trl==0.11.4"]:
    r = pip(["install", "-q", "--no-cache-dir", p])
    print(("OK" if r.returncode==0 else "ERR"), p)

pip(["uninstall", "-y", "bitsandbytes"])
print("✅ bitsandbytes removed")

for sp in site.getsitepackages():
    bnb_file = Path(sp) / "peft" / "tuners" / "lora" / "bnb.py"
    if bnb_file.exists():
        bnb_file.write_text(
            "# patched\n"
            "def dispatch_bnb_8bit(*a, **k): return None\n"
            "def dispatch_bnb_4bit(*a, **k): return None\n"
        )
        print("✅ peft/bnb.py patched")
        break

for sp in site.getsitepackages():
    tv_dir = Path(sp) / "torchvision"
    if not tv_dir.exists(): continue
    (tv_dir / "extension.py").write_text(
        "_HAS_OPS = False\n"
        "def _has_ops(): return False\n"
        "def _check_cuda_version(): pass\n"
        "def _load_library(n): return None\n"
        "def _assert_has_ops(): return None\n"
    )
    (tv_dir / "_meta_registrations.py").write_text("# patched\n")
    (tv_dir / "__init__.py").write_text(
        '__version__ = "0.0.0-patched"\n'
        "from .extension import _HAS_OPS\n"
        "try:\n    from . import transforms\nexcept: transforms = None\n"
        "try:\n    from . import io\nexcept: io = None\n"
    )
    print("✅ torchvision patched")
    break

print("\n⚠️  RESTART THE KERNEL now!")

OK transformers @ git
OK huggingface-hub==0.28.1
OK accelerate==1.3.0
OK peft==0.13.2
OK trl==0.11.4
✅ bitsandbytes removed
✅ peft/bnb.py patched
✅ torchvision patched

⚠️  RESTART THE KERNEL now!


## 2. Version check

## 3. Configuration and dataset

In [ ]:
import warnings, logging, torch, peft, torchvision, accelerate, transformers
warnings.filterwarnings('ignore')
logging.getLogger('transformers').setLevel(logging.ERROR)
logging.getLogger('peft').setLevel(logging.ERROR)

print(f"torch        : {torch.__version__}")
print(f"transformers : {transformers.__version__}")
print(f"peft         : {peft.__version__}")
print(f"accelerate   : {accelerate.__version__}")

from transformers import AutoProcessor, AutoConfig
p = AutoProcessor.from_pretrained(
    "zai-org/GLM-OCR", trust_remote_code=True)
print(f"✅ AutoProcessor : {type(p).__name__}")
c = AutoConfig.from_pretrained(
    "zai-org/GLM-OCR", trust_remote_code=True)
print(f"✅ model_type    : {c.model_type}")
del p, c
print("✅ All good")

torch        : 2.10.0+cu128
transformers : 5.10.0.dev0
peft         : 0.13.2
accelerate   : 1.3.0
✅ AutoProcessor : Glm46VProcessor
✅ model_type    : glm_ocr
✅ All good


In [ ]:
import os, json, subprocess, torch
from pathlib import Path

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["OMP_NUM_THREADS"]        = "1"

# ── Session s14 ───────────────────────────────────────────────
KAGGLE_USERNAME = '<KAGGLE_USER>'
SESSION_ID      = 'ldc_v1s14'

# ── Resume checkpoints (local epoch 51) ───────────────────────
RESUME_CKPT   = Path('/kaggle/input/datasets/<KAGGLE_USER>/resumeep51')
BEST_CKPT     = Path('/kaggle/input/datasets/<KAGGLE_USER>/resumeep51')
START_EPOCH   = 51
BEST_LOSS_S13 = 0.0352
NO_IMPROVE    = 12

# ── Input images + annotations ────────────────────────────────
BASE_DATA  = Path('/kaggle/input/datasets/<KAGGLE_USER>/lettredechangeglm'
                  '/lettredechange_glm_zhai')
TRAIN_DIR  = BASE_DATA / 'train'
TEST_DIR   = BASE_DATA / 'test'
TRAIN_JSON = BASE_DATA / 'train_glmocr.json'
TEST_JSON  = BASE_DATA / 'test_glmocr.json'

WORK_DIR   = Path('/kaggle/working')
OUTPUT_DIR = WORK_DIR / 'glmocr_ldc_finetuned'
BEST_DIR   = WORK_DIR / 'glmocr_ldc_best'
OUTPUT_DIR.mkdir(exist_ok=True)
BEST_DIR.mkdir(exist_ok=True)

MODEL_NAME = 'zai-org/GLM-OCR'

NUM_EPOCHS              = 100
EARLY_STOPPING_PATIENCE = 20
BATCH_SIZE              = 1
GRAD_ACCUM              = 8
LEARNING_RATE           = 2e-6
LORA_RANK               = 4
LORA_ALPHA              = 8
SEED                    = 42
MAX_SIZE                = 1024
MAX_SEQ_LEN             = 2048

ENTITIES = [
    'numero_traite', 'date_echeance', 'montant',
    'tireur', 'tire', 'rib', 'domiciliation'
]

gpu_ok = torch.cuda.is_available()
print(f"GPU : {torch.cuda.get_device_name(0) if gpu_ok else 'CPU only'}")

IMAGE_INDEX = {}
for folder in [TRAIN_DIR, TEST_DIR]:
    if folder.exists():
        for p in folder.rglob('*'):
            if p.suffix.lower() in ['.jpg', '.jpeg', '.png']:
                IMAGE_INDEX[p.name] = p
print(f"Indexed images: {len(IMAGE_INDEX)}")

def fix_paths(data, name):
    fixed, ko = [], 0
    for item in data:
        img = item["images"][0]
        if Path(img).exists():
            fixed.append(item); continue
        img_name = img.replace("\\", "/").split("/")[-1]
        if img_name in IMAGE_INDEX:
            it = dict(item)
            it["images"] = [str(IMAGE_INDEX[img_name])]
            fixed.append(it)
        else:
            ko += 1
    print(f"[{name}] OK={len(fixed)} KO={ko}")
    return fixed

train_data  = json.loads(Path(TRAIN_JSON).read_text())
test_data   = json.loads(Path(TEST_JSON).read_text())
train_fixed = fix_paths(train_data, "TRAIN")
test_fixed  = fix_paths(test_data,  "TEST")

DATASET_WORK = WORK_DIR / "dataset_ldc"
DATASET_WORK.mkdir(exist_ok=True)
(DATASET_WORK/"train_ldc.json").write_text(
    json.dumps(train_fixed, ensure_ascii=False), encoding="utf-8")
(DATASET_WORK/"test_ldc.json").write_text(
    json.dumps(test_fixed, ensure_ascii=False), encoding="utf-8")

print(f"Train={len(train_fixed)} Test={len(test_fixed)}")
r = subprocess.run(["df","-h","/kaggle/working"],
                   capture_output=True, text=True)
print(f"💾 Disk:\n{r.stdout}")
print(f'\n📋 Config {SESSION_ID} :')
print(f'   START_EPOCH={START_EPOCH} BEST_LOSS={BEST_LOSS_S13}')
print(f'   NO_IMPROVE={NO_IMPROVE}/{EARLY_STOPPING_PATIENCE}')
print(f'   LEARNING_RATE={LEARNING_RATE}')
print(f'   KAGGLE_USERNAME={KAGGLE_USERNAME}')
print(f'   → early stopping in {EARLY_STOPPING_PATIENCE-NO_IMPROVE} epochs without improvement')

assert RESUME_CKPT.exists(), f"❌ Resume introuvable : {RESUME_CKPT}"
print(f"\n✅ Resume OK : {RESUME_CKPT}")

## 4. Prompt
Position-guided prompt (French and Arabic field labels) describing where each field is located on the document.

In [ ]:
PROMPT = """Tu es un expert OCR spécialisé dans les lettres de change tunisiennes (كمبيالة).
Analyse l'image et extrais les 7 champs suivants en te guidant par leur POSITION sur le document :

1. numero_traite
   → Coin HAUT DROIT du document, après "Ordre de paiement L-CN" ou "أمر بدفع الكمبيالة عدد"
   → Suite de chiffres (10-12 chiffres), aussi visible dans le code-barres en BAS
   → Exemple : 012345678901

2. date_echeance
   → Zone HAUT GAUCHE, champ "Echéance / حلول الأجل"
   → C'est la date FUTURE (pas la date de création)
   → Format JJ/MM/AAAA
   → Exemple : 15/09/2026

3. montant
   → Zone HAUT DROIT, champ "Montant / البلغ"
   → Valeur numérique avec point ou virgule décimale
   → Ignorer "Montant en lettres"
   → Exemple : 2000.000

4. tireur
   → Zone MILIEU GAUCHE, champ "Tireur / الساحب"
   → Nom de la société ou personne qui émet la traite
   → Souvent en MAJUSCULES ou tampon
   → Exemple : SOCIETE EXEMPLE SARL

5. tire
   → Zone BAS CENTRE, champ "Nom et adresse du Tiré / اسم وعنوان المسحوب عليه"
   → Prendre UNIQUEMENT la première ligne (nom/société), ignorer l'adresse
   → Exemple : SOCIETE CLIENTE

6. rib
   → Zone MILIEU, champ "RIB ou RIP du Tiré / المعرف البنكي"
   → Code étab + Code Agence + N° Compte + Clé, sans espaces, chiffres uniquement
   → Exemple : 01234567890123456789

7. domiciliation
   → Zone BAS DROIT, champ "Domiciliation / اسم وعنوان الفرع المسحوب عليه"
   → Nom de la banque uniquement (BIAT, BNA, STB, Attijari, UIB, BTK...)
   → Exemple : BIAT

RÈGLES STRICTES :
- Répondre UNIQUEMENT avec un objet JSON valide, aucun texte avant ou après
- Si un champ est illisible → "illisible"
- Si un champ est absent → "N/A"
- Pour le RIB : chiffres uniquement, ignorer les lettres parasites

FORMAT :
{"numero_traite": "...", "date_echeance": "...", "montant": "...", "tireur": "...", "tire": "...", "rib": "...", "domiciliation": "..."}"""

print(f"✅ Prompt defined — {len(PROMPT)} characters")

✅ Prompt defined — 1866 characters


## 5. Checkpointing callback
Saves the best adapter and a resumable checkpoint after every epoch.

In [ ]:
import time, json, shutil, subprocess
from pathlib import Path
import torch
from transformers import TrainerCallback

def upload_to_kaggle(folder_path, dataset_name):
    folder = Path(folder_path)
    folder.mkdir(parents=True, exist_ok=True)
    metadata = {
        "title"    : dataset_name,
        "id"       : f"{KAGGLE_USERNAME}/{dataset_name}",
        "licenses" : [{"name": "CC0-1.0"}]
    }
    with open(folder / 'dataset-metadata.json', 'w') as f:
        json.dump(metadata, f)
    r = subprocess.run(
        ['kaggle', 'datasets', 'create',
         '-p', str(folder), '--dir-mode', 'zip'],
        capture_output=True, text=True, timeout=300
    )
    if r.returncode == 0:
        print(f'   ☁️  Created: {dataset_name}'); return
    r2 = subprocess.run(
        ['kaggle', 'datasets', 'version',
         '-p', str(folder), '-m', 'update',
         '--dir-mode', 'zip'],
        capture_output=True, text=True, timeout=300
    )
    print(f'   {"☁️  Updated" if r2.returncode==0 else "⚠️  Failed"}'
          f' : {dataset_name}')


class KaggleSaver(TrainerCallback):
    def __init__(self, output_dir, best_dir, patience=20,
                 max_hours=11.0, resume_every_epochs=1):
        self.output_dir          = Path(output_dir)
        self.best_dir            = Path(best_dir)
        self.processor           = None
        self.patience            = patience
        self.no_improve          = 0
        self.best_score          = float("inf")
        self.best_epoch          = 0
        self.start_time          = time.time()
        self.max_seconds         = max_hours * 3600
        self.resume_every_epochs = resume_every_epochs
        self.last_resume_epoch   = -1
        self.emergency_done      = False

    def _latest_checkpoint(self):
        cps = sorted(self.output_dir.glob("checkpoint-*"))
        return cps[-1] if cps else None

    def zip_best(self, epoch, model):
        self.best_dir.mkdir(parents=True, exist_ok=True)
        model.save_pretrained(str(self.best_dir))
        if self.processor:
            self.processor.save_pretrained(str(self.best_dir))
        for old in Path("/kaggle/working").glob(
                f"ldc-{SESSION_ID}-best_*.zip"):
            old.unlink()
        name = f"ldc-{SESSION_ID}-best-ep{epoch:02d}"
        shutil.make_archive(
            f"/kaggle/working/{name}", "zip", str(self.best_dir))
        sz = Path(f"/kaggle/working/{name}.zip").stat().st_size/1e6
        print(f"📦 {name}.zip ({sz:.0f} MB) [BEST]")
        upload_to_kaggle(str(self.best_dir), name)

    def zip_resume(self, epoch):
        cp = self._latest_checkpoint()
        if cp is None:
            print("⚠️ No checkpoint."); return
        (cp/"callback_state.json").write_text(json.dumps({
            "best_score": self.best_score,
            "best_epoch": self.best_epoch,
            "no_improve": self.no_improve,
            "epoch"     : epoch,
            "session_id": SESSION_ID,
        }, indent=2))
        for old in Path("/kaggle/working").glob(
                f"ldc-{SESSION_ID}-resume_*.zip"):
            old.unlink()
        name     = f"ldc-{SESSION_ID}-resume-ep{epoch:02d}"
        zip_path = f"/kaggle/working/{name}.zip"
        shutil.make_archive(
            f"/kaggle/working/{name}", "zip", str(cp))
        sz = Path(zip_path).stat().st_size/1e6
        print(f"📦 {name}.zip ({sz:.0f} MB) [RESUME]")
        resume_dir = Path(f"/kaggle/working/resume_ep{epoch:02d}")
        if resume_dir.exists(): shutil.rmtree(resume_dir)
        resume_dir.mkdir()
        shutil.copy(zip_path, resume_dir / f"{name}.zip")
        upload_to_kaggle(str(resume_dir), name)

    def on_evaluate(self, args, state, control,
                    metrics=None, model=None, **kwargs):
        if not metrics or model is None: return
        epoch     = int(state.epoch or 0)
        eval_loss = metrics.get("eval_loss")
        if eval_loss is not None:
            if eval_loss < self.best_score:
                self.best_score = eval_loss
                self.best_epoch = epoch
                self.no_improve = 0
                print(f"✅ BEST eval_loss={eval_loss:.4f} epoch={epoch}")
                self.zip_best(epoch, model)
            else:
                self.no_improve += 1
                print(f"⏳ ({self.no_improve}/{self.patience}) "
                      f"eval_loss={eval_loss:.4f} "
                      f"best={self.best_score:.4f}")
        elapsed   = time.time() - self.start_time
        remaining = self.max_seconds - elapsed
        if remaining < 3600 and not self.emergency_done:
            print(f"🚨 Emergency save ({remaining/60:.0f} min restantes)")
            self.zip_resume(epoch)
            self.emergency_done = True
        if self.no_improve >= self.patience:
            print("🛑 Early stopping.")
            self.zip_resume(epoch)
            control.should_training_stop = True
        torch.cuda.empty_cache()

    def on_save(self, args, state, control, **kwargs):
        epoch = int(state.epoch or 0)
        if (epoch > 0
                and epoch % self.resume_every_epochs == 0
                and self.last_resume_epoch != epoch):
            self.zip_resume(epoch)
            self.last_resume_epoch = epoch
        for cp in sorted(
                self.output_dir.glob("checkpoint-*"))[:-1]:
            shutil.rmtree(cp, ignore_errors=True)
        return control


disk_saver = KaggleSaver(
    OUTPUT_DIR, BEST_DIR,
    patience            = EARLY_STOPPING_PATIENCE,
    max_hours           = 11.0,
    resume_every_epochs = 1
)
disk_saver.best_score = BEST_LOSS_S13
disk_saver.best_epoch = START_EPOCH
disk_saver.no_improve = NO_IMPROVE
print(f"✅ KaggleSaver s14 ready")
print(f"   best_score : {disk_saver.best_score}")
print(f"   best_epoch : {disk_saver.best_epoch}")
print(f"   no_improve : {disk_saver.no_improve}/{EARLY_STOPPING_PATIENCE}")
print(f"   → early stopping in {EARLY_STOPPING_PATIENCE-NO_IMPROVE} epochs")

## 6. Model, LoRA adapters and tokenised cache

In [ ]:
import warnings, logging, torch
from pathlib import Path
from peft import PeftModel
from transformers import AutoProcessor, AutoModelForImageTextToText

warnings.filterwarnings('ignore')
logging.getLogger('transformers').setLevel(logging.ERROR)
logging.getLogger('peft').setLevel(logging.ERROR)

import peft
bnb_file = Path(peft.__file__).parent / 'tuners' / 'lora' / 'bnb.py'
if bnb_file.exists():
    bnb_file.write_text(
        '# disabled\n'
        'def dispatch_bnb_8bit(*a, **k): return None\n'
        'def dispatch_bnb_4bit(*a, **k): return None\n'
    )
import peft.tuners.lora.model as lora_model
lora_model.is_bnb_available      = lambda: False
lora_model.is_bnb_4bit_available = lambda: False
print('✅ bitsandbytes disabled')

torch.cuda.empty_cache()

print(f'⏳ Loading the processor...')
processor = AutoProcessor.from_pretrained(
    MODEL_NAME, use_fast=False, trust_remote_code=True)
print(f'✅ Processor : {type(processor).__name__}')
disk_saver.processor = processor

print(f'\n⏳ Loading the base model {MODEL_NAME}...')
base_model = AutoModelForImageTextToText.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16 if gpu_ok else torch.float32,
    device_map='auto' if gpu_ok else None,
    trust_remote_code=True,
)
base_model.config.use_cache = False
print(f'✅ Base: {sum(p.numel() for p in base_model.parameters()):,} params')

print(f'\n⏳ Loading LoRA from {RESUME_CKPT.name}...')
model = PeftModel.from_pretrained(
    base_model,
    str(RESUME_CKPT),
    is_trainable=True
)
model.enable_input_require_grads()
print('✅ gradient checkpointing DISABLED for resuming')

model.print_trainable_parameters()
print(f'\n✅ Model s14 ready — resuming from epoch {START_EPOCH}')
print(f'   LR={LEARNING_RATE} | no_improve={NO_IMPROVE}/20')

✅ bitsandbytes disabled
⏳ Loading the processor...
✅ Processor : Glm46VProcessor

⏳ Loading the base model zai-org/GLM-OCR...


Loading weights:   0%|          | 0/510 [00:00<?, ?it/s]

✅ Base: 1,107,405,824 params

⏳ Loading LoRA from resumeep51...
✅ gradient checkpointing DISABLED for resuming
trainable params: 2,727,936 || all params: 1,110,133,760 || trainable%: 0.2457

✅ Model s14 ready — resuming from epoch 51
   LR=2e-06 | no_improve=12/20


In [ ]:
import warnings, logging, torch, json, gc
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings('ignore')
logging.getLogger('transformers').setLevel(logging.ERROR)

CACHE_INPUT = Path('/kaggle/input/datasets/<KAGGLE_USER>/cacheons')
CACHE_WORK  = Path('/kaggle/working/tokens_cache_ldc')

if CACHE_INPUT.exists() and \
        list((CACHE_INPUT/'train').glob('*.pt')):
    CACHE_DIR = CACHE_INPUT
    print(f'✅ Cache from INPUT: {CACHE_DIR}')
else:
    print(f'⏳ Building the cache in {CACHE_WORK}...')
    CACHE_DIR = CACHE_WORK
    CACHE_WORK.mkdir(exist_ok=True)
    (CACHE_WORK/'train').mkdir(exist_ok=True)
    (CACHE_WORK/'test').mkdir(exist_ok=True)

    def preprocess_and_save(data, split_name):
        split_dir = CACHE_DIR / split_name
        split_dir.mkdir(exist_ok=True)
        saved, errors = 0, 0
        for idx, item in enumerate(data):
            if idx % 200 == 0:
                r = subprocess.run(
                    ['df','--output=avail','-BM',
                     '/kaggle/working'],
                    capture_output=True, text=True)
                avail = r.stdout.strip().split('\n')[-1].strip()
                print(f'   [{split_name}] {idx}/{len(data)} '
                      f'libre={avail}')
            out_file = split_dir / f'{idx:05d}.pt'
            if out_file.exists():
                saved += 1; continue
            answer        = item['conversations'][1]['content']
            orig_img_path = item['images'][0]
            try:
                image = Image.open(orig_img_path).convert('RGB')
                w, h  = image.size
                if max(w, h) > MAX_SIZE:
                    ratio = MAX_SIZE / max(w, h)
                    image = image.resize(
                        (int(w*ratio), int(h*ratio)), Image.LANCZOS)
            except:
                image = Image.new('RGB', (224,224), 'white')
            messages = [
                {"role": "user", "content": [
                    {"type": "image", "image": image},
                    {"type": "text",  "text":  PROMPT},
                ]},
                {"role": "assistant", "content": [
                    {"type": "text", "text": answer},
                ]},
            ]
            try:
                import subprocess as sp
                inputs = processor.apply_chat_template(
                    messages, tokenize=True,
                    add_generation_prompt=False,
                    return_dict=True, return_tensors='pt',
                    processor_kwargs={
                        "return_mm_token_type_ids": True,
                        "truncation": False, "padding": False,
                    },
                )
                inputs.pop('token_type_ids', None)
                input_ids = inputs['input_ids'].squeeze(0)
                attn      = inputs['attention_mask'].squeeze(0)
                mm_ids    = inputs.get('mm_token_type_ids', None)
                if mm_ids is not None:
                    mm_ids = mm_ids.squeeze(0)
                prompt_inputs = processor.apply_chat_template(
                    [messages[0]], tokenize=True,
                    add_generation_prompt=True,
                    return_dict=True, return_tensors='pt',
                    processor_kwargs={
                        "return_mm_token_type_ids": True,
                        "truncation": False, "padding": False,
                    },
                )
                prompt_len = prompt_inputs['input_ids'].shape[1]
                labels = input_ids.clone()
                labels[:min(prompt_len, labels.numel())] = -100
                sample = {
                    'input_ids'      : input_ids.cpu(),
                    'attention_mask' : attn.cpu(),
                    'labels'         : labels.cpu(),
                    'orig_image_path': orig_img_path,
                }
                if mm_ids is not None:
                    sample['mm_token_type_ids'] = mm_ids.cpu()
                if 'image_grid_thw' in inputs:
                    sample['image_grid_thw'] = \
                        inputs['image_grid_thw'].squeeze(0).cpu()
                torch.save(sample, out_file)
                saved += 1
                del inputs, input_ids, attn, labels, sample
                if idx % 100 == 0: gc.collect()
            except Exception as e:
                errors += 1
                if errors <= 5:
                    print(f'   ⚠️  idx={idx} : {str(e)[:100]}')
        print(f'   [{split_name}] ✅ saved={saved} errors={errors}')
        return saved

    import subprocess
    print(f'⏳ Cache train ({len(train_fixed)} examples)...')
    preprocess_and_save(train_fixed, 'train')
    print(f'\n⏳ Cache test ({len(test_fixed)} exemples)...')
    preprocess_and_save(test_fixed, 'test')
    print(f'\n✅ Cache built in {CACHE_DIR}')

train_pts = list((CACHE_DIR/'train').glob('*.pt'))
test_pts  = list((CACHE_DIR/'test').glob('*.pt'))
print(f'   train: {len(train_pts)} fichiers')
print(f'   test : {len(test_pts)} fichiers')

class CachedDataset(Dataset):
    def __init__(self, cache_dir, split, processor):
        self.files     = sorted(
            (Path(cache_dir)/split).glob('*.pt'))
        self.processor = processor
        print(f'✅ {split} : {len(self.files)} fichiers')

    def __len__(self): return len(self.files)

    def __getitem__(self, idx):
        sample        = torch.load(
            self.files[idx], weights_only=False)
        orig_img_path = sample.pop('orig_image_path', None)
        if orig_img_path:
            try:
                image = Image.open(orig_img_path).convert('RGB')
                w, h  = image.size
                if max(w, h) > MAX_SIZE:
                    ratio = MAX_SIZE / max(w, h)
                    image = image.resize(
                        (int(w*ratio), int(h*ratio)), Image.LANCZOS)
                msgs = [{"role": "user", "content": [
                    {"type": "image", "image": image},
                    {"type": "text",  "text":  PROMPT},
                ]}]
                pv = self.processor.apply_chat_template(
                    msgs, tokenize=True,
                    add_generation_prompt=True,
                    return_dict=True, return_tensors='pt',
                    processor_kwargs={
                        "return_mm_token_type_ids": True,
                        "truncation": False, "padding": False,
                    },
                )
                if 'pixel_values' in pv:
                    sample['pixel_values'] = \
                        pv['pixel_values'].squeeze(0)
                if 'image_grid_thw' in pv:
                    sample['image_grid_thw'] = \
                        pv['image_grid_thw'].squeeze(0)
            except:
                pass
        return sample

def collate_fn(batch):
    b   = batch[0]
    out = {}
    for k, v in b.items():
        if isinstance(v, torch.Tensor):
            if k in ['input_ids', 'labels',
                     'attention_mask', 'mm_token_type_ids']:
                if v.shape[0] > MAX_SEQ_LEN:
                    v = v[:MAX_SEQ_LEN]
            out[k] = v.unsqueeze(0)
    return out

train_dataset = CachedDataset(CACHE_DIR, 'train', processor)
test_dataset  = CachedDataset(CACHE_DIR, 'test',  processor)

print('🔍 Test DataLoader...')
batch = next(iter(DataLoader(
    train_dataset, batch_size=1,
    num_workers=0, collate_fn=collate_fn
)))
for k, v in batch.items():
    print(f'   {k:<25} : {v.shape} {v.dtype}')
n_visible = (batch['labels'][0] != -100).sum().item()
print(f'\n   Labels visibles : {n_visible}')
assert 'mm_token_type_ids' in batch, '❌ mm_token_type_ids manquant'
assert 'pixel_values'      in batch, '❌ pixel_values manquant'
assert n_visible > 0,                '❌ Aucun label visible'
print(f'\n✅ DataLoader OK')
del batch
torch.cuda.empty_cache()

✅ Cache from INPUT: /kaggle/input/datasets/<KAGGLE_USER>/cacheons
   train: 4258 fichiers
   test : 1065 fichiers
✅ train : 4258 fichiers
✅ test : 1065 fichiers
🔍 Test DataLoader...
   input_ids                 : torch.Size([1, 1351]) torch.int64
   attention_mask            : torch.Size([1, 1351]) torch.int64
   labels                    : torch.Size([1, 1351]) torch.int64
   mm_token_type_ids         : torch.Size([1, 1351]) torch.int64
   image_grid_thw            : torch.Size([1, 3]) torch.int64
   pixel_values              : torch.Size([1, 2116, 1176]) torch.float32

   Labels visibles : 103

✅ DataLoader OK


## 7. Training (resumed session)

In [ ]:
import os, shutil, json, warnings, logging, gc
from pathlib import Path
from transformers import TrainingArguments, Trainer, set_seed
import torch

warnings.filterwarnings('ignore')
logging.getLogger('transformers').setLevel(logging.ERROR)
set_seed(SEED)

os.environ["WANDB_DISABLED"]         = "true"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_ALLOC_CONF"]     = "expandable_segments:True"
if gpu_ok:
    torch.backends.cuda.matmul.allow_tf32 = True

gc.collect()
torch.cuda.empty_cache()
free = torch.cuda.get_device_properties(0).total_memory/1e9 \
     - torch.cuda.memory_allocated()/1e9
print(f'🖥️  Free VRAM: {free:.1f}GB')

disk_saver.best_score     = BEST_LOSS_S13
disk_saver.best_epoch     = START_EPOCH
disk_saver.no_improve     = NO_IMPROVE
disk_saver.processor      = processor
disk_saver.emergency_done = False
disk_saver.start_time     = __import__('time').time()

model.print_trainable_parameters()

class LowMemTrainer(Trainer):
    def training_step(self, model, inputs,
                      num_items_in_batch=None):
        torch.cuda.empty_cache()
        return super().training_step(
            model, inputs, num_items_in_batch)

training_args = TrainingArguments(
    output_dir                  = str(OUTPUT_DIR),
    num_train_epochs            = NUM_EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = 1,
    gradient_accumulation_steps = GRAD_ACCUM,
    learning_rate               = LEARNING_RATE,
    warmup_steps                = 0,
    lr_scheduler_type           = 'cosine',
    weight_decay                = 0.05,
    neftune_noise_alpha         = 5,
    fp16                        = gpu_ok,
    max_grad_norm               = 1.0,
    logging_steps               = 20,
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    save_total_limit            = 1,
    load_best_model_at_end      = False,
    report_to                   = 'none',
    seed                        = SEED,
    remove_unused_columns       = False,
    dataloader_num_workers      = 0,
    dataloader_pin_memory       = False,
    gradient_checkpointing      = False,
    optim                       = 'adamw_torch_fused',
)

trainer = LowMemTrainer(
    model         = model,
    args          = training_args,
    train_dataset = train_dataset,
    eval_dataset  = test_dataset,
    data_collator = collate_fn,
    callbacks     = [disk_saver],
)

print(f'\n🚀 Fine-tuning GLM-OCR s14 — {SESSION_ID}')
print(f'   Account         : {KAGGLE_USERNAME}')
print(f'   Train={len(train_dataset)} Test={len(test_dataset)}')
print(f'   LR={LEARNING_RATE} rank={LORA_RANK} MAX_SIZE={MAX_SIZE}px')
print(f'   NEFTune=5 weight_decay=0.05 warmup_steps=0')
print(f'   gradient_checkpointing=False ✅')
print(f'   Resuming from epoch {START_EPOCH+1}')
print(f'   best_loss={BEST_LOSS_S13} no_improve={NO_IMPROVE}/20')
print(f'   → early stopping in {EARLY_STOPPING_PATIENCE-NO_IMPROVE} epochs')
print(f'   Auto upload: ✅ ({KAGGLE_USERNAME})')

try:
    trainer.train(
        resume_from_checkpoint=str(RESUME_CKPT))

except torch.cuda.OutOfMemoryError:
    print('\n❌ OOM')
    trainer.model.save_pretrained(str(BEST_DIR))
    processor.save_pretrained(str(BEST_DIR))
    upload_to_kaggle(str(BEST_DIR),
                     f'ldc-{SESSION_ID}-oom-backup')
    raise

except Exception as e:
    print(f'\n❌ Crash : {e}')
    trainer.model.save_pretrained(str(BEST_DIR))
    processor.save_pretrained(str(BEST_DIR))
    upload_to_kaggle(str(BEST_DIR),
                     f'ldc-{SESSION_ID}-crash-backup')
    raise

processor.save_pretrained(str(BEST_DIR))
upload_to_kaggle(
    str(BEST_DIR),
    f'ldc-{SESSION_ID}-final-ep{disk_saver.best_epoch:02d}'
)
print(f'\n✅ Finished {SESSION_ID}!')
print(f'   Best : {disk_saver.best_score:.4f} '
      f'(epoch {disk_saver.best_epoch})')
print(f'   Kaggle   : kaggle.com/datasets/{KAGGLE_USERNAME}/'
      f'ldc-{SESSION_ID}-final-ep{disk_saver.best_epoch:02d}')

🖥️  Free VRAM: 14.8GB
trainable params: 2,727,936 || all params: 1,110,133,760 || trainable%: 0.2457

🚀 Fine-tuning GLM-OCR s14 — ldc_v1s14
   Account         : <KAGGLE_USER>
   Train=4258 Test=1065
   LR=2e-06 rank=4 MAX_SIZE=1024px
   NEFTune=5 weight_decay=0.05 warmup_steps=0
   gradient_checkpointing=False ✅
   Resuming from epoch 52
   best_loss=0.0352 no_improve=12/20
   → early stopping in 8 epochs
   Auto upload: ✅ (<KAGGLE_USER>)
{'loss': '0.001723', 'grad_norm': '0.2111', 'learning_rate': '2.419e-06', 'epoch': '51.03'}
{'loss': '0.001277', 'grad_norm': '0.5327', 'learning_rate': '2.416e-06', 'epoch': '51.07'}
{'loss': '0.002013', 'grad_norm': '0.7423', 'learning_rate': '2.413e-06', 'epoch': '51.11'}
{'loss': '0.00355', 'grad_norm': '3.357', 'learning_rate': '2.41e-06', 'epoch': '51.14'}
{'loss': '0.004169', 'grad_norm': '1.379', 'learning_rate': '2.407e-06', 'epoch': '51.18'}
{'loss': '0.002732', 'grad_norm': '3.147', 'learning_rate': '2.404e-06', 'epoch': '51.22'}
{'loss': '